In [17]:
import os
from pathlib import Path


# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = "/Users/conradlz/Documents/webclones/freqtrade"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())

/Users/conradlz/Documents/webclones/freqtrade


In [18]:
from freqtrade.configuration.configuration import Configuration

config = Configuration.from_files(["user_data/config.json"])
backtest_dir = config["user_data_dir"] / "backtest_results"

# Define some constants
config["timeframe"] = "5m"
# Name of the strategy class
config["strategy"] = "WarriorMomentum"
# Location of the data
data_location = config["datadir"]
# Pair to analyze - Only use one pair here
pair = "XMR/USDT"

2025-07-21 14:19:36,826 - freqtrade.configuration.load_config - INFO - Using config: user_data/config.json ...

2025-07-21 14:19:36,836 - freqtrade.loggers - INFO - Enabling colorized output.

2025-07-21 14:19:36,838 - freqtrade.loggers - INFO - Logfile configured

2025-07-21 14:19:36,839 - freqtrade.loggers - INFO - Verbosity set to 0

2025-07-21 14:19:36,841 - freqtrade.configuration.configuration - INFO - Using user-data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data ...

2025-07-21 14:19:36,842 - freqtrade.configuration.configuration - INFO - Using data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken ...

2025-07-21 14:19:36,844 - freqtrade.exchange.check_exchange - INFO - Checking exchange...

2025-07-21 14:19:36,854 - freqtrade.exchange.check_exchange - INFO - Exchange "kraken" is officially supported by the Freqtrade development team.

In [19]:
# Load data using values set above
from freqtrade.data.history import load_pair_history


candles = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format="feather",  # Make sure to update this to your data
)

# Confirm success
print(f"Loaded {len(candles)} rows of data for {pair} from {data_location}")
candles.head()

2025-07-21 14:19:36,899 - freqtrade.data.converter.converter - INFO - Missing data fillup for XMR/USDT, 5m: before: 5795 - after: 8903 - 53.63%

Loaded 8903 rows of data for XMR/USDT from /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken


,date,open,high,low,close,volume
0,2025-06-16 21:25:00+00:00,328.75,328.75,328.74,328.74,1.022868
1,2025-06-16 21:30:00+00:00,327.16,327.16,327.14,327.14,7.000000
2,2025-06-16 21:35:00+00:00,328.59,328.67,327.82,327.82,1.524258
3,2025-06-16 21:40:00+00:00,328.08,328.09,328.08,328.09,2.732000
4,2025-06-16 21:45:00+00:00,328.09,328.09,328.09,328.09,0.000000


In [21]:

from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats
from freqtrade.plot.plotting import generate_candlestick_graph


# Limit graph period to keep plotly quick and reactive

# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()

# Filter trades to one pair
trades_red = trades.loc[trades["pair"] == pair]

# Use candles data instead of undefined 'data' variable
data_red = candles
# Generate candlestick graph
graph = generate_candlestick_graph(
    pair=pair,
    data=data_red,
    trades=trades_red,
    indicators1=["ema9", "ema21", "ema50", "bb_upperband", "bb_lowerband", "bb_middleband"],
    indicators2=["rsi", "macd", "macdsignal", "macdhist", "adx", "mfi"],
    plot_config={'show_date': True}
)

# Show graph inline
graph.show()

# Render graph in a separate window
# graph.show(renderer="browser")

2025-07-21 14:24:17,809 - freqtrade.data.btanalysis.bt_fileutils - INFO - Loading backtest result from 
/Users/conradlz/Documents/webclones/freqtrade/user_data/backtest_results/backtest-result-2025-07-20_10-48-22.zip

2025-07-21 14:24:17,895 - freqtrade.plot.plotting - INFO - Indicator "ema9" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,908 - freqtrade.plot.plotting - INFO - Indicator "ema21" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,910 - freqtrade.plot.plotting - INFO - Indicator "ema50" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,911 - freqtrade.plot.plotting - INFO - Indicator "bb_middleband" ignored. Reason: This indicator is not found in your strategy.

...

2025-07-21 14:24:17,948 - freqtrade.plot.plotting - INFO - Indicator "macd" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,950 - freqtrade.plot.plotting - INFO - Indicator "macdsignal" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,952 - freqtrade.plot.plotting - INFO - Indicator "macdhist" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,955 - freqtrade.plot.plotting - INFO - Indicator "adx" ignored. Reason: This indicator is not found in your strategy.

2025-07-21 14:24:17,961 - freqtrade.plot.plotting - INFO - Indicator "mfi" ignored. Reason: This indicator is not found in your strategy.